# 05 - EDA and Feature Engineering

**Objective:** Explore Breast Cancer data patterns and create new features.

**Steps:**
1. Statistical summary and class distribution
2. Visualizations (distributions, correlations, feature comparison by diagnosis)
3. Feature scaling
4. Feature engineering (interaction and ratio features)
5. Save engineered data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully")

In [ ]:
# TODO: Load the clean data and inspect its structure
# Start by reading data/processed/clean_data.csv into a DataFrame.
# Then use .info() to see column dtypes and non-null counts,
# .head() to preview a few rows, and .describe() for summary statistics.

PROCESSED_DIR = Path("../data/processed")

# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
# print(df.info())
# print(df.head())
# print(df.describe())

### Statistical Summary & Distribution

Before building any models, it is important to understand the range and spread of your data.
The target variable — `Diagnosis` — is binary (a classification problem).
Check the class balance to see if the dataset is imbalanced, which might affect model evaluation.

Breast cancer data has 30 features grouped into three categories:
- **Mean_** (10 features): average measurements of cell nuclei
- **SE_** (10 features): standard error of measurements
- **Worst_** (10 features): largest (worst) measurements

In [ ]:
# TODO: Examine the target variable distribution and feature summary
# Use df.describe() to see min, max, mean, and quartiles for all features.
# Check the class balance with df["Diagnosis"].value_counts().
#
# print(df.describe())
# print(df["Diagnosis"].value_counts())

#### A little primer on groupby 

- `groupby` is a powerful pandas method that allows you to split your data into groups based on some criteria, apply a function to each group, and then combine the results. For example, to see how measurements differ between benign and malignant tumors, you can do:

```python
mean_by_diagnosis = df.groupby("Diagnosis")["Mean_Radius"].mean()
```

- `aggregate` is a method that allows you to apply multiple functions to your grouped data. For example, to get both the mean and standard deviation of Mean_Radius by Diagnosis, you can do:

```python
stats_by_diagnosis = df.groupby("Diagnosis")["Mean_Radius"].aggregate(["mean", "std"])
```

Aggregate functions can be any function that takes a Series and returns a single value, such as `mean`, `std`, `min`, `max`, etc.
Aggregate can be deployed on multiple columns at once, and you can specify different functions for each column if needed.

In [ ]:
# === Executed Example: GroupBy and Aggregate ===
# Small inline dataset showing how groupby splits data by Diagnosis
# and compares average measurements between classes.

import pandas as pd

data = pd.DataFrame({
    "Diagnosis": [0, 0, 0, 1, 1, 1],
    "Mean_Radius": [12.0, 13.5, 11.8, 17.2, 18.5, 16.9],
    "Mean_Texture": [18.5, 20.2, 17.1, 21.4, 22.0, 20.8],
})

mean_by_diag = data.groupby("Diagnosis")["Mean_Radius"].mean()
print("Average Mean_Radius by Diagnosis:\n", mean_by_diag)

stats_by_diag = data.groupby("Diagnosis")["Mean_Texture"].agg(["mean", "std", "min", "max"])
print("\nTexture statistics by Diagnosis:\n", stats_by_diag)

In [ ]:
# === Commented Template: GroupBy and Aggregate ===
# Uncomment and adapt to your own dataset.

# import pandas as pd
# data = pd.DataFrame({
#     "group_col": [val1, val1, val2, val2],
#     "value_col": [10, 20, 30, 40],
# })
# grouped = data.groupby("group_col")["value_col"].mean()
# stats = data.groupby("group_col")["value_col"].agg(["mean", "std", "min", "max"])

### Missing Value Imputation

Our clean data has no missing values, but the corrupted variants from
`data_injection/` do. Common strategies:

- **Drop rows**: `df.dropna()` — fast, loses samples
- **Mean/Median imputation**: `SimpleImputer(strategy='median')` — preserves sample count
- **KNN imputation**: `KNNImputer()` — estimates from neighbors, more robust
- **Forward fill**: `df.ffill()` — for sequential data

In [ ]:
# === Executed Example: Missing Value Imputation ===
# Using SimpleImputer to fill missing values with the mean.

from sklearn.impute import SimpleImputer

data = pd.DataFrame({
    "Mean_Radius": [12.0, np.nan, 11.8, 17.2, np.nan],
    "Mean_Texture": [18.5, 20.2, np.nan, 21.4, 22.0],
})

print("Before imputation:")
print(data)

imputer = SimpleImputer(strategy='mean')
imputed = imputer.fit_transform(data)
imputed_df = pd.DataFrame(imputed, columns=data.columns)
print("\nAfter imputation (mean strategy):")
print(imputed_df)
print(f"\nImputed Mean_Radius: {imputer.statistics_[0]:.3f}")
print(f"Imputed Mean_Texture: {imputer.statistics_[1]:.3f}")

In [ ]:
# TODO: Compare feature distributions by Diagnosis
# Which features best separate benign (0) from malignant (1)?
# Use boxplots to compare the top discriminative features.
#
# features_to_plot = ["Mean_Radius", "Mean_Texture", "Mean_Area", "Mean_Concave_Points"]
# fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# for ax, feature in zip(axes.ravel(), features_to_plot):
#     sns.boxplot(x="Diagnosis", y=feature, data=df, ax=ax)
#     ax.set_title(f"{feature} by Diagnosis")
# plt.tight_layout()
# plt.show()

### Visualizations

Visual exploration helps you spot patterns and relationships that summary statistics miss.
Focus on:
- How each feature is distributed (histograms)
- How features correlate with each other and with the target (heatmap)
- Which features separate benign from malignant (boxplots)

In [ ]:
# TODO: Plot histograms for key numerical features
# Pick a handful of Mean_ features that are likely to differ between classes.
# Use df[features].hist() with bins=30 and a large figure size.

# plt.tight_layout()
# plt.show()

In [ ]:
# TODO: Create a correlation heatmap
# Use df.corr() to compute pairwise correlations between numeric columns.
# Then visualize with sns.heatmap() — this quickly shows which features
# are strongly (positively or negatively) correlated with each other.
#
# TODO: Identify highly correlated features (|corr| > 0.9)
# The breast cancer dataset has many correlated features (e.g., Mean_Radius
# and Mean_Perimeter are nearly perfectly correlated). This is multicollinearity
# and can affect models like logistic regression.

# mean_features = [col for col in df.columns if col.startswith("Mean_")]
# plt.figure(figsize=(12, 10))
# sns.heatmap(df[mean_features + ["Diagnosis"]].corr(), annot=False, cmap="coolwarm", center=0)
# plt.title("Mean Feature Correlation Heatmap")
# plt.show()
#
# target_corr = df.corr()["Diagnosis"].sort_values(ascending=False)
# print("Top correlations with Diagnosis:")
# print(target_corr)

### Feature Scaling

Many machine learning algorithms (SVR, linear models, neural networks) are sensitive to
the scale of input features. StandardScaler transforms each feature to have mean 0 and
standard deviation 1, which puts all features on equal footing.

Tree-based models (Random Forest, XGBoost) do not require scaling since they split on
thresholds independently of feature magnitude.

In [ ]:
# === Executed Example: Feature Scaling ===
# Small inline dataset showing how StandardScaler transforms features
# to have mean ~0 and std ~1.

from sklearn.preprocessing import StandardScaler
import pandas as pd

data = pd.DataFrame({
    "Mean_Radius": [12.0, 13.5, 11.8, 17.2, 18.5],
    "Mean_Texture": [18.5, 20.2, 17.1, 21.4, 22.0],
    "Diagnosis": [0, 0, 0, 1, 1],
})

scaler = StandardScaler()
scaled_features = scaler.fit_transform(data[["Mean_Radius", "Mean_Texture"]])
scaled_df = pd.DataFrame(scaled_features, columns=["Mean_Radius_scaled", "Mean_Texture_scaled"])
scaled_df["Diagnosis"] = data["Diagnosis"]
print(scaled_df)
print(f"Means after scaling: {scaled_df[['Mean_Radius_scaled', 'Mean_Texture_scaled']].mean().values}")
print(f"Stds after scaling: {scaled_df[['Mean_Radius_scaled', 'Mean_Texture_scaled']].std().values}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# TODO: Separate features and target, then scale the features
# Define X as all columns except "Diagnosis" and y as the target column.
#
# Create a StandardScaler, call fit_transform() on X to compute the mean and std
# and return the scaled array in one step.
#
# After scaling, verify that each feature has mean ~0 and std ~1.

# X = df.drop("Diagnosis", axis=1)
# y = df["Diagnosis"]
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)
# print(f"Mean after scaling (first 5): {X_scaled.mean(axis=0)[:5]}")
# print(f"Std after scaling (first 5): {X_scaled.std(axis=0)[:5]}")
# print(f"All means near zero: {np.allclose(X_scaled.mean(axis=0), 0, atol=1e-10)}")

### Feature Engineering

New features derived from existing columns can capture interactions and non-linear relationships.
Good candidates for breast cancer:
- **Ratio features**: `Mean_Radius / Mean_Perimeter` — tumor shape compactness
- **Interaction features**: `Mean_Radius × Mean_Concave_Points` — combined malignancy indicator
- **Area ratio**: `Worst_Area / Mean_Area` — growth rate proxy

Be careful with division by zero — add a small epsilon or +1 to the denominator.

#### Note

In pandas, you can create interaction features like this:

```python
df["feature1_feature2"] = df["feature1"] * df["feature2"]
```

In [ ]:
# === Executed Example: Interaction Features ===
# Multiplication and ratio on a small inline dataset.

import pandas as pd

data = pd.DataFrame({
    "Mean_Radius": [12.0, 13.5, 11.8, 17.2, 18.5],
    "Mean_Perimeter": [78.5, 85.2, 74.1, 112.4, 120.0],
    "Mean_Concave_Points": [0.05, 0.08, 0.03, 0.15, 0.18],
})

data["Radius_Perimeter_Ratio"] = data["Mean_Radius"] / data["Mean_Perimeter"]
print("Radius / Perimeter ratio:\n", data[["Mean_Radius", "Mean_Perimeter", "Radius_Perimeter_Ratio"]])

data["Radius_Concavity"] = data["Mean_Radius"] * data["Mean_Concave_Points"]
print("\nRadius × Concavity:\n", data[["Mean_Radius", "Mean_Concave_Points", "Radius_Concavity"]])

In [ ]:
# === Commented Template: Interaction Features ===
# Uncomment and adapt to your own dataset.

# import pandas as pd
# data = pd.DataFrame({
#     "feature_a": [val1, val2, val3],
#     "feature_b": [val1, val2, val3],
# })
# data["a_times_b"] = data["feature_a"] * data["feature_b"]
# EPS = 1e-6
# data["a_over_b"] = data["feature_a"] / (data["feature_b"] + EPS)

In [ ]:
# TODO: Create new features based on domain knowledge
# The breast cancer dataset has three measurement groups (Mean_, SE_, Worst_).
# Ratio features within each group can capture shape properties.
#
# Examples:
#   df["Radius_Perimeter_Ratio"] = df["Mean_Radius"] / df["Mean_Perimeter"]
#   df["Texture_Radius_Ratio"] = df["Mean_Texture"] / df["Mean_Radius"]
#
# TODO: Create interaction features across groups
# Malignant tumors tend to have large Worst_ values relative to Mean_ values.
#   df["Worst_to_Mean_Area"] = df["Worst_Area"] / df["Mean_Area"]
#   df["Worst_to_Mean_Radius"] = df["Worst_Radius"] / df["Mean_Radius"]
#
# TODO: Verify the new features
# Check that they have finite values and reasonable ranges with .describe().

In [ ]:
# TODO: Save the engineered data for the next notebook
# Include both the original and new features.

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# df.to_csv(PROCESSED_DIR / "engineered_data.csv", index=False)
# print("Engineered data saved to data/processed/engineered_data.csv")

### Unsupervised Clustering (KMeans)

Clustering groups observations without using the target labels.
We use **KMeans** which partitions data into $k$ clusters by minimizing
within-cluster variance.

**Questions:**
- Do the clusters found by KMeans align with the actual Diagnosis classes?
- How many natural groups exist in the data?


In [ ]:
# TODO: Apply KMeans clustering and compare with target labels
# Clustering groups data without using labels.
# Use the elbow method to find optimal k, then compare clusters vs target.

# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import adjusted_rand_score
# from sklearn.decomposition import PCA

# Step 1: Scale features
# X_clust = df.drop("Diagnosis", axis=1).select_dtypes(include=[np.number])
# X_clust_scaled = StandardScaler().fit_transform(X_clust)
#
# Step 2: Elbow method for k=2..10
# inertias = []
# for k in range(2, min(11, X_clust_scaled.shape[1] + 1)):
#     kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
#     kmeans.fit(X_clust_scaled)
#     inertias.append(kmeans.inertia_)
#
# Step 3: Plot elbow curve
# plt.plot(range(2, min(11, X_clust_scaled.shape[1] + 1)), inertias, 'bo-')
# plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow Method')
# plt.show()
#
# Step 4: Fit KMeans and compare with target
# df["Cluster"] = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_clust_scaled)
# ari = adjusted_rand_score(df["Diagnosis"], df["Cluster"])
# print(f"Adjusted Rand Index: {ari:.4f}")
# print(pd.crosstab(df["Cluster"], df["Diagnosis"]))
#
# Step 5: Visualize via PCA
# pca_vis = PCA(n_components=2, random_state=42)
# X_pca_vis = pca_vis.fit_transform(X_clust_scaled)
# plt.scatter(X_pca_vis[:, 0], X_pca_vis[:, 1], c=df['Cluster'], cmap='viridis', edgecolors='k', alpha=0.7)
# plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('KMeans Clusters')
# plt.show()


### Principal Component Analysis (PCA)

PCA finds orthogonal axes (principal components) that capture the maximum variance
in the data. It is useful for:
- **Dimensionality reduction**: compressing many features into fewer components
- **Visualization**: projecting high-dimensional data to 2D or 3D
- **Noise reduction**: discarding low-variance components

PCA is **unsupervised** — it does not use the target labels.


In [ ]:
# TODO: Apply PCA for dimensionality reduction and visualization
# PCA finds the axes of maximum variance in the data.
# It can reduce high-dimensional data to 2D for visualization.

# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler

# X = df.drop("Diagnosis", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
#
# Step 1: Fit PCA with min(n_features, 10) components
# n_comps = min(X.shape[1], 10)
# pca = PCA(n_components=n_comps, random_state=42)
# X_pca = pca.fit_transform(X_scaled)
#
# Step 2: Scree plot
# plt.bar(range(1, n_comps + 1), pca.explained_variance_ratio_)
# plt.xlabel('PC'); plt.ylabel('Explained Variance Ratio')
# plt.title('Scree Plot'); plt.show()
#
# Step 3: Cumulative explained variance
# cumulative = np.cumsum(pca.explained_variance_ratio_)
# plt.plot(range(1, n_comps + 1), cumulative, 'bo-')
# plt.axhline(y=0.95, color='r', linestyle='--', label='95%')
# plt.legend(); plt.show()
#
# Step 4: 2D PCA scatter colored by target
# plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["Diagnosis"], cmap="coolwarm", edgecolors="k", alpha=0.7)
# plt.xlabel('PC1'); plt.ylabel('PC2')
# plt.title('PCA Projection'); plt.colorbar()
# plt.show()
#
# Step 5: Inspect feature loadings
# loadings = pd.DataFrame(
#     pca.components_.T[:, :3],
#     index=X.columns,
#     columns=['PC1', 'PC2', 'PC3'],
# )
# print(loadings['PC1'].abs().sort_values(ascending=False).head(5))


### Linear Discriminant Analysis (LDA)

LDA finds axes that **maximize class separability**. Unlike PCA (unsupervised),
LDA uses the target labels to find projections that best separate the classes.

**LDA vs PCA**:
- PCA maximizes **variance** (ignores labels)
- LDA maximizes **class separation** (uses labels)
- For classification, LDA often gives better separation in fewer components


In [ ]:
# TODO: Apply LDA for supervised dimensionality reduction
# LDA uses the target labels to maximize class separation.
# Compare its projection with PCA (unsupervised).

# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# X = df.drop("Diagnosis", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
# y = df["Diagnosis"]
#
# Step 1: Fit LDA
# n_classes = y.nunique()
# lda = LDA(n_components=min(n_classes - 1, X_scaled.shape[1]))
# X_lda = lda.fit_transform(X_scaled, y)
# print(f"LDA reduced to {X_lda.shape[1]} component(s)")
#
# Step 2: Visualize
# if X_lda.shape[1] >= 2:
#     plt.scatter(X_lda[:, 0], X_lda[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
#     plt.xlabel('LD1'); plt.ylabel('LD2')
# else:
#     for cls in sorted(y.unique()):
#         plt.hist(X_lda[y == cls, 0], bins=20, alpha=0.5, label=f'Class {cls}')
#     plt.legend(); plt.xlabel('LD1'); plt.ylabel('Frequency')
# plt.title('LDA Projection'); plt.show()
#
# Step 3: Side-by-side PCA vs LDA
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
# ax1.set_title('PCA (unsupervised)'); ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')
# x_lda_2d = X_lda[:, :min(2, X_lda.shape[1])]
# if x_lda_2d.shape[1] == 1:
#     x_lda_2d = np.hstack([x_lda_2d, np.zeros_like(x_lda_2d)])
# ax2.scatter(x_lda_2d[:, 0], x_lda_2d[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
# ax2.set_title('LDA (supervised)'); ax2.set_xlabel('LD1'); ax2.set_ylabel('LD2')
# plt.tight_layout(); plt.show()


### Recursive Feature Elimination (RFE)

RFE recursively removes the least important features, building a model at each step.
It ranks features by importance and finds the optimal subset.

**Benefits:**
- Reduces overfitting by removing noisy features
- Improves model interpretability
- Can speed up training and prediction


In [ ]:
# TODO: Apply RFE for feature selection
# RFE recursively removes the least important features.
# RFECV uses cross-validation to find the optimal subset.

# from sklearn.feature_selection import RFE, RFECV
# from sklearn.linear_model import LogisticRegression

# X = df.drop("Diagnosis", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
# y = df["Diagnosis"]
#
# Step 1: Fit RFE
# estimator = LogisticRegression(max_iter=1000, random_state=42)
# rfe = RFE(estimator=estimator, n_features_to_select=min(10, X_scaled.shape[1]), step=1)
# rfe.fit(X_scaled, y)
#
# Step 2: Display feature rankings
# ranking_df = pd.DataFrame({
#     'Feature': X.columns,
#     'Rank': rfe.ranking_,
#     'Selected': rfe.support_,
# }).sort_values('Rank')
# print(ranking_df)
#
# Step 3: RFECV for automatic feature count
# rfecv = RFECV(estimator=estimator, step=1, cv=5, scoring='accuracy')
# rfecv.fit(X_scaled, y)
# print(f"Optimal features: {rfecv.n_features_}")
#
# Step 4: Plot CV accuracy vs feature count
# plt.plot(range(len(rfecv.cv_results_['mean_test_score'])),
#          rfecv.cv_results_['mean_test_score'], 'bo-')
# plt.axvline(x=rfecv.n_features_, color='r', linestyle='--')
# plt.title('RFE: Optimal Feature Count'); plt.show()


### Exercises

1. **Try different scalers**: Replace `StandardScaler` with `MinMaxScaler` or `RobustScaler` and compare how the scaled distributions look.
2. **Identify redundant features**: Find pairs of features with correlation > 0.95. Would dropping one affect model performance?
3. **Create a mean-to-se ratio**: For each Mean_ feature, create a corresponding `Mean_to_SE` ratio (e.g., `df["Radius_Mean_to_SE"] = df["Mean_Radius"] / (df["Radius_SE"] + 1e-6)`). Does this capture measurement reliability?
4. **Log transform skewed features**: Some features like `Area` may be right-skewed. Try `np.log1p()` and check if the distribution becomes more normal.
5. **Pairplot**: Use `sns.pairplot()` on a subset of the most discriminative features, coloring by Diagnosis — do benign and malignant samples form distinct clusters?